# Chapter 10: Codes (Error Correcting Codes) - Examples

Jupyter notebook demonstrating Hamming distance, Hamming codes, CRC, and Reed-Solomon codes.

## 1. Hamming distance and weight

$d(x,y)$ = number of positions where they differ.

In [ ]:
def hamming_distance(x, y):
    return sum(1 for a, b in zip(x, y) if a != b)

a = [1, 0, 1, 1, 0]
b = [1, 1, 0, 1, 0]
print(f"d({a}, {b}) = {hamming_distance(a, b)}")
weight = sum(a)
print(f"wt({a}) = {weight}  ( = d(a, 0) )")

## 2. Hamming code [7,4]

Encodes 4 data bits into 7 codeword bits and corrects any single error. Its minimum distance is 3, so it corrects 1 error.

In [ ]:
# Hamming(7,4) generator matrix (data bits d3 d2 d1 d0 -> 7 bits)
def hamming74_encode(data):  # data = [d0,d1,d2,d3]
    d = list(data)
    return [
        d[0]^d[1]^d[3],  # p1
        d[0]^d[2]^d[3],  # p2
        d[0],
        d[1]^d[2]^d[3],  # p3
        d[1], d[2], d[3]
    ]

def hamming74_syndrome(codeword):  # decode syndrome
    c = list(codeword)
    s1 = c[3]^c[4]^c[5]^c[6]
    s2 = c[1]^c[2]^c[5]^c[6]
    s3 = c[0]^c[2]^c[4]^c[6]
    return (s3<<2)|(s2<<1)|s1

data = [1,0,1,1]
cw = hamming74_encode(data)
print("Codeword:", cw)
err = list(cw); err[2] ^= 1  # flip one bit
print("Corrupted:", err)
syndrome = hamming74_syndrome(err)
print(f"Syndrome = {syndrome} (indicates bit position to correct)")
print(f"Decoded data: {[err[i] for i in [2,4,5,6]]}")

## 3. CRC (Cyclic Redundancy Check)

Polynomial codes over GF(2). Compute CRC-8 of a byte string.

In [ ]:
def crc8(data, poly=0x07):
    crc = 0
    for byte in data:
        crc ^= byte
        for _ in range(8):
            crc = ((crc << 1) ^ (poly if crc & 0x80 else 0)) & 0xFF
    return crc

msg = b"hello"
checksum = crc8(msg)
print(f"CRC-8 of {msg!r} = {checksum:#04x}")
print(f"CRC-8 of corrupted msg = {crc8(b'hella'):#04x} (differs)")

## 4. Reed-Solomon encoding

Reed-Solomon codes work over finite fields and can correct up to $t = (n-k)/2$ symbol errors. Evaluate a message polynomial at $n$ points.

In [ ]:
# Encode message m(x) = m0 + m1 x + ... over GF(2^8)
def gf256_mul(a, b, mod=0x11B):
    result = 0
    low = mod & 0xFF
    while b:
        if b & 1:
            result ^= a
        carry = a & 0x80
        a = (a << 1) & 0xFF
        if carry:
            a ^= low
        b >>= 1
    return result

def rs_encode(msg, alpha=2, n=255):
    """Simplified RS evaluation-style encoding: evaluate polynomial at powers of alpha."""
    # msg = coefficients, m(x) = sum msg[i] x^i
    codeword = []
    for i in range(n):
        x = pow(alpha, i, 256) if False else alpha**i
        # evaluate poly at x using Horner in GF(256)
        val = 0
        for c in reversed(msg):
            val = gf256_mul(val, x) ^ c
        codeword.append(val)
    return codeword

# Small demo over GF with n small
msg = [1, 2, 3]  # message polynomial 1 + 2x + 3x^2
cw = rs_encode(msg, n=7)
print("RS encoded symbols:", cw)